# DDIM Interpolation with SDXL Base

Generates one canonical man and one canonical woman using **SDXL Base 1.0**,
then interpolates between them using the DDIM paper method:

```
man_img  -> VAE encode -> x0_man  -> DDIM invert -> z_T_man  --+
                                                                +-slerp(a)-> z_T_i -> DDIM denoise -> x0 -> VAE decode -> image
woman_img -> VAE encode -> x0_woman -> DDIM invert -> z_T_woman -+
```

Everything uses **one model** (SDXL Base) so the noise space is consistent.
Prompt embeddings are also lerped at each step for semantic consistency.

## Configuration

In [ ]:
OUTPUT_DIR   = "output/ddim_interpolation"

N_INTERP     = 100   # number of interpolation steps (including endpoints)
N_DDIM_STEPS = 50    # DDIM inversion / denoising steps
N_GEN_STEPS  = 50    # generation steps
GEN_CFG      = 7.5   # guidance scale for generation
N_COPIES     = 50    # copies of endpoint latents (for downstream DPS)

MAN_SEED     = 0
WOMAN_SEED   = 1
GLOBAL_SEED  = 42

MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"

MAN_PROMPT = (
    "a hyperrealistic studio portrait photograph of a very masculine man, "
    "strong jawline, short hair, formal attire, sharp features, "
    "professional photography, 8k"
)
WOMAN_PROMPT = (
    "a hyperrealistic studio portrait photograph of a very feminine woman, "
    "long hair, soft features, elegant attire, beautiful, "
    "professional photography, 8k"
)

print(f"MODEL        : {MODEL_ID}")
print(f"N_INTERP     : {N_INTERP}")
print(f"N_DDIM_STEPS : {N_DDIM_STEPS}")
print(f"N_GEN_STEPS  : {N_GEN_STEPS}  CFG={GEN_CFG}")
print(f"OUTPUT_DIR   : {OUTPUT_DIR}")

## 1 - Colab Setup

In [ ]:
import os, sys

if 'google.colab' in str(get_ipython()):
    get_ipython().system('pip install -q diffusers transformers accelerate xformers')
    get_ipython().system('pip install -q scikit-learn matplotlib Pillow tqdm')
    try:
        from google.colab import userdata
        from huggingface_hub import login
        hf_token = userdata.get('HF')
        login(token=hf_token) if hf_token else login()
        print('HF login OK')
    except Exception as e:
        print(f'HF login skipped: {e}')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output dir: {os.path.abspath(OUTPUT_DIR)}')

## 2 - Imports

In [ ]:
import copy, json, pickle
import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision.transforms.functional as TF
from diffusers import StableDiffusionXLPipeline, DDIMScheduler
from sklearn.decomposition import PCA
from tqdm.notebook import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
torch.manual_seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

## 3 - Load SDXL Base with DDIM Scheduler

In [ ]:
print(f'Loading {MODEL_ID} ...')
pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant='fp16',
).to(device)

# Must use DDIM for deterministic inversion
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe.set_progress_bar_config(disable=True)

SF = pipe.vae.config.scaling_factor
print(f'Scheduler : {type(pipe.scheduler).__name__}')
print(f'VAE sf    : {SF}')
print('Models loaded.')

## 4 - Helper Functions

In [ ]:
# ── Generation ────────────────────────────────────────────────────────────────
def generate_one(prompt, seed, n_steps=N_GEN_STEPS, cfg=GEN_CFG):
    """
    Generate one 512x512 image with SDXL Base.
    Returns (PIL image, latent [4,64,64] float32 scaled z*sf).
    """
    generator = torch.Generator(device=device).manual_seed(seed)
    with torch.no_grad():
        result = pipe(
            prompt=prompt,
            height=512, width=512,
            num_inference_steps=n_steps,
            guidance_scale=cfg,
            output_type='pil',
            return_dict=True,
            generator=generator,
        )
        pil = result.images[0]

        # VAE encode in fp32 to avoid overflow/NaN
        pipe.vae.to(torch.float32)
        img_t  = TF.to_tensor(pil).unsqueeze(0).to(device).float()
        img_t  = (img_t * 2.0) - 1.0
        latent = pipe.vae.encode(img_t).latent_dist.mean
        latent = latent * SF
        pipe.vae.to(torch.float16)

    if torch.isnan(latent).any() or torch.isinf(latent).any():
        raise RuntimeError(f'NaN/Inf in latent for seed={seed}.')
    return pil, latent[0].float().cpu()


# ── VAE decode ────────────────────────────────────────────────────────────────
def decode_latent(lat):
    """Single [4,64,64] or [1,4,64,64] float32 scaled latent -> PIL."""
    if lat.dim() == 3:
        lat = lat.unsqueeze(0)
    pipe.vae.to(torch.float32)
    with torch.no_grad():
        out = pipe.vae.decode(lat.float().to(device) / SF).sample
        out = torch.clamp((out + 1.0) / 2.0, 0.0, 1.0)
    pipe.vae.to(torch.float16)
    return TF.to_pil_image(out[0].cpu())


def decode_latents_batch(lats, batch_size=4):
    """[N,4,64,64] float32 scaled latents -> list of PIL images."""
    pipe.vae.to(torch.float32)
    images = []
    with torch.no_grad():
        for i in range(0, len(lats), batch_size):
            b   = lats[i:i+batch_size].float().to(device)
            out = pipe.vae.decode(b / SF).sample
            out = torch.clamp((out + 1.0) / 2.0, 0.0, 1.0)
            for j in range(out.shape[0]):
                images.append(TF.to_pil_image(out[j].cpu()))
    pipe.vae.to(torch.float16)
    return images


# ── Prompt encoding ───────────────────────────────────────────────────────────
def encode_prompt(prompt):
    """Returns (encoder_states, added_cond_kwargs) for use in UNet."""
    with torch.no_grad():
        pe, ne, ppool, npool = pipe.encode_prompt(
            prompt=prompt, negative_prompt='',
            device=device, do_classifier_free_guidance=True,
            num_images_per_prompt=1,
        )
    enc  = torch.cat([ne, pe], dim=0)
    tids = torch.tensor([[512,512,0,0,512,512]], dtype=pe.dtype, device=device)
    ack  = {'text_embeds': torch.cat([npool, ppool], dim=0),
            'time_ids':    tids.repeat(2,1)}
    return enc, ack, pe, ne, ppool, npool


# ── DDIM Inversion ────────────────────────────────────────────────────────────
def ddim_invert(x0_lat, prompt, n_steps=N_DDIM_STEPS):
    """
    DDIM inversion: x0 latent (scaled) -> z_T noise.
    Deterministic forward ODE: t=0 -> t=T.
    x0_lat : [1, 4, 64, 64] float16 on device
    Returns: [1, 4, 64, 64] float16
    """
    enc, ack, _, _, _, _ = encode_prompt(prompt)

    sched = copy.deepcopy(pipe.scheduler)
    sched.set_timesteps(n_steps)
    timesteps = sched.timesteps.flip(0)   # low t -> high t
    alphas    = sched.alphas_cumprod.to(device)

    xt = x0_lat.clone().half()
    with torch.no_grad():
        for idx, t in enumerate(timesteps):
            noise_pred = pipe.unet(
                torch.cat([xt]*2), t,
                encoder_hidden_states=enc,
                added_cond_kwargs=ack,
                return_dict=False,
            )[0]
            eps = noise_pred.chunk(2)[0]   # unconditional only

            a_t    = alphas[t.long()]
            t_next = timesteps[min(idx+1, len(timesteps)-1)]
            a_next = (alphas[t_next.long()]
                      if idx+1 < len(timesteps)
                      else torch.tensor(0.0, device=device))

            x0_pred = (xt - (1-a_t).sqrt() * eps) / a_t.sqrt()
            xt = a_next.sqrt() * x0_pred + (1-a_next).sqrt() * eps
    return xt


# ── DDIM Denoising ────────────────────────────────────────────────────────────
def ddim_denoise(z_T, enc, ack, n_steps=N_DDIM_STEPS):
    """
    DDIM denoising: z_T -> x0 latent (scaled).
    Accepts pre-computed enc/ack so we can pass interpolated embeddings.
    z_T : [1, 4, 64, 64] float16 on device
    Returns: [1, 4, 64, 64] float16
    """
    sched = copy.deepcopy(pipe.scheduler)
    sched.set_timesteps(n_steps)

    xt = z_T.clone().half()
    with torch.no_grad():
        for t in sched.timesteps:
            noise_pred = pipe.unet(
                torch.cat([xt]*2), t,
                encoder_hidden_states=enc,
                added_cond_kwargs=ack,
                return_dict=False,
            )[0]
            eps = noise_pred.chunk(2)[0]
            xt  = sched.step(eps, t, xt, return_dict=False)[0]
    return xt


# ── Slerp ─────────────────────────────────────────────────────────────────────
def slerp(v0, v1, t):
    v0f   = v0.reshape(-1).float()
    v1f   = v1.reshape(-1).float()
    dot   = torch.dot(v0f/v0f.norm(), v1f/v1f.norm()).clamp(-1,1)
    omega = torch.acos(dot)
    if omega.abs() < 1e-6:
        return ((1-t)*v0 + t*v1).to(v0.dtype)
    return ((torch.sin((1-t)*omega)/torch.sin(omega))*v0 +
            (torch.sin(   t*omega)/torch.sin(omega))*v1).to(v0.dtype)


# ── Display ───────────────────────────────────────────────────────────────────
def show_row(images, titles=None, w=3):
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(w*n, w+0.4))
    if n == 1: axes = [axes]
    for i,(ax,img) in enumerate(zip(axes, images)):
        ax.imshow(img); ax.axis('off')
        if titles: ax.set_title(titles[i], fontsize=8)
    plt.tight_layout(); plt.show()

print('Helpers defined.')

## 5 - Generate Canonical Portraits

In [ ]:
print(f'Generating canonical man  (seed={MAN_SEED}, {N_GEN_STEPS} steps)...')
man_pil, man_latent = generate_one(MAN_PROMPT, seed=MAN_SEED)
man_pil.save(os.path.join(OUTPUT_DIR, 'man_canonical.png'))
print(f'  norm={man_latent.norm():.3f}')
show_row([man_pil], titles=[f'Man  seed={MAN_SEED}'])

In [ ]:
print(f'Generating canonical woman (seed={WOMAN_SEED}, {N_GEN_STEPS} steps)...')
woman_pil, woman_latent = generate_one(WOMAN_PROMPT, seed=WOMAN_SEED)
woman_pil.save(os.path.join(OUTPUT_DIR, 'woman_canonical.png'))
print(f'  norm={woman_latent.norm():.3f}')
show_row([woman_pil], titles=[f'Woman  seed={WOMAN_SEED}'])

In [ ]:
mf = man_latent.reshape(-1).float()
wf = woman_latent.reshape(-1).float()
print(f'Cosine sim : {torch.dot(mf/mf.norm(), wf/wf.norm()).item():.4f}')
print(f'L2 dist    : {(man_latent - woman_latent).norm().item():.3f}')
show_row([man_pil, woman_pil], titles=['Man', 'Woman'])

## 6 - DDIM Inversion

In [ ]:
man_x0   = man_latent.unsqueeze(0).half().to(device)
woman_x0 = woman_latent.unsqueeze(0).half().to(device)

print(f'Inverting man   ({N_DDIM_STEPS} steps)...')
z_T_man = ddim_invert(man_x0, MAN_PROMPT)
print(f'  z_T_man   norm={z_T_man.norm():.3f}  (pure N(0,1) ~{(4*64*64)**0.5:.0f})')

print(f'Inverting woman ({N_DDIM_STEPS} steps)...')
z_T_woman = ddim_invert(woman_x0, WOMAN_PROMPT)
print(f'  z_T_woman norm={z_T_woman.norm():.3f}')
print('Inversion done.')

In [ ]:
# Verify: denoise z_T_man back -> should recover man_pil
print('Verifying reconstruction...')
enc_m, ack_m, _, _, _, _ = encode_prompt(MAN_PROMPT)
enc_w, ack_w, _, _, _, _ = encode_prompt(WOMAN_PROMPT)

man_rec_pil   = decode_latent(ddim_denoise(z_T_man,   enc_m, ack_m).float())
woman_rec_pil = decode_latent(ddim_denoise(z_T_woman, enc_w, ack_w).float())

show_row(
    [man_pil, man_rec_pil, woman_pil, woman_rec_pil],
    titles=['Original man', 'Reconstructed man', 'Original woman', 'Reconstructed woman'],
    w=3.5
)
print('Good reconstruction = DDIM inversion is working correctly.')

## 7 - Slerp in Noise Space + DDIM Denoise
Both the noise vector and the prompt embedding are interpolated at each step.

In [ ]:
# Pre-encode both prompts once
_, _, pe_m, ne_m, pp_m, np_m = encode_prompt(MAN_PROMPT)
_, _, pe_w, ne_w, pp_w, np_w = encode_prompt(WOMAN_PROMPT)

alphas = torch.linspace(0.0, 1.0, N_INTERP)
interp_latents_list = []

print(f'Generating {N_INTERP} interpolated images ({N_DDIM_STEPS} DDIM steps each)...')
for alpha in tqdm(alphas, desc='DDIM interp'):
    a = alpha.item()

    # 1. Slerp noise vectors
    z_T_i = slerp(z_T_man, z_T_woman, a)

    # 2. Lerp prompt embeddings
    pe_i  = (1-a)*pe_m  + a*pe_w
    ne_i  = (1-a)*ne_m  + a*ne_w
    pp_i  = (1-a)*pp_m  + a*pp_w
    np_i  = (1-a)*np_m  + a*np_w
    enc_i = torch.cat([ne_i, pe_i], dim=0)
    tids  = torch.tensor([[512,512,0,0,512,512]], dtype=pe_i.dtype, device=device)
    ack_i = {'text_embeds': torch.cat([np_i, pp_i], dim=0),
             'time_ids':    tids.repeat(2,1)}

    # 3. DDIM denoise with interpolated noise + interpolated prompt
    x0_i = ddim_denoise(z_T_i, enc_i, ack_i)
    interp_latents_list.append(x0_i.squeeze(0).float().cpu())

interp_latents = torch.stack(interp_latents_list, dim=0)   # [N_INTERP, 4, 64, 64]
torch.save(interp_latents, os.path.join(OUTPUT_DIR, 'interp_vae_latents.pt'))
print(f'interp_vae_latents.pt -> {tuple(interp_latents.shape)}')

## 8 - Decode All Frames

In [ ]:
print(f'Decoding {N_INTERP} frames...')
interp_pil = decode_latents_batch(interp_latents, batch_size=4)

decoded_dir = os.path.join(OUTPUT_DIR, 'interp_decoded')
os.makedirs(decoded_dir, exist_ok=True)
for i, img in enumerate(tqdm(interp_pil, desc='Saving')):
    a_str = f'{alphas[i].item():.3f}'.replace('.','p')
    img.save(os.path.join(decoded_dir, f'interp_{i:03d}_a{a_str}.png'))
print(f'{len(interp_pil)} frames -> {decoded_dir}')

## 9 - Contact Sheet

In [ ]:
kf = np.linspace(0, N_INTERP-1, 10, dtype=int)
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('DDIM Interpolation: Masculine Man -> Feminine Woman  (SDXL Base)',
             fontsize=13, fontweight='bold')
for j, idx in enumerate(kf):
    ax = axes[j//5][j%5]
    ax.imshow(interp_pil[idx])
    ax.set_title(f'step {idx}  a={alphas[idx]:.2f}', fontsize=9)
    ax.axis('off')
plt.tight_layout()
contact_path = os.path.join(OUTPUT_DIR, 'interp_contact_sheet.png')
fig.savefig(contact_path, dpi=110, bbox_inches='tight')
plt.show()
print(f'Contact sheet -> {contact_path}')

## 10 - Save Endpoint Latents + PCA + Metadata

In [ ]:
# Endpoint latents x N_COPIES (for downstream DPS target distribution)
man_lats   = interp_latents[0].unsqueeze(0).expand(N_COPIES,-1,-1,-1).clone()
woman_lats = interp_latents[-1].unsqueeze(0).expand(N_COPIES,-1,-1,-1).clone()
torch.save(man_lats,   os.path.join(OUTPUT_DIR, 'man_vae_latents.pt'))
torch.save(woman_lats, os.path.join(OUTPUT_DIR, 'woman_vae_latents.pt'))
print(f'man_vae_latents.pt   -> {tuple(man_lats.shape)}')
print(f'woman_vae_latents.pt -> {tuple(woman_lats.shape)}')

# PCA
flat = interp_latents.reshape(N_INTERP, -1).numpy()
pca  = PCA(n_components=2)
crd  = pca.fit_transform(flat)
var  = pca.explained_variance_ratio_.sum()

fig, ax = plt.subplots(figsize=(10,5))
sc = ax.scatter(crd[:,0], crd[:,1], c=np.arange(N_INTERP), cmap='RdBu_r',
                s=60, edgecolors='black', linewidths=0.3)
plt.colorbar(sc, ax=ax, label='Step')
ax.plot(crd[:,0], crd[:,1], 'k--', alpha=0.25, lw=1)
ax.scatter(*crd[0],  s=250, c='royalblue', marker='*', label='Man')
ax.scatter(*crd[-1], s=250, c='crimson',   marker='*', label='Woman')
for i in range(0, N_INTERP, max(1, N_INTERP//10)):
    ax.annotate(str(i), crd[i], xytext=(4,4), textcoords='offset points', fontsize=7)
ax.set_title(f'VAE Latent PCA (DDIM interp)  Var={var:.1%}')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
pca_path = os.path.join(OUTPUT_DIR, 'interp_pca.png')
fig.savefig(pca_path, dpi=130, bbox_inches='tight')
plt.show()

with open(os.path.join(OUTPUT_DIR, 'pca_latent.pkl'), 'wb') as f:
    pickle.dump(pca, f)

# Metadata
meta = {
    'method': 'ddim_inversion_slerp_noise_plus_prompt_lerp',
    'model_id': MODEL_ID,
    'n_gen_steps': N_GEN_STEPS, 'gen_cfg': GEN_CFG,
    'n_ddim_steps': N_DDIM_STEPS, 'n_interp': N_INTERP, 'n_copies': N_COPIES,
    'man_seed': MAN_SEED, 'woman_seed': WOMAN_SEED,
    'z_T_man_norm': z_T_man.norm().item(),
    'z_T_woman_norm': z_T_woman.norm().item(),
    'pca_var': float(var),
    'man_prompt': MAN_PROMPT, 'woman_prompt': WOMAN_PROMPT,
}
with open(os.path.join(OUTPUT_DIR, 'metadata.json'), 'w') as f:
    json.dump(meta, f, indent=2)
print('All outputs saved.')

## 11 - Summary

In [ ]:
print('='*60)
print('DDIM Interpolation complete')
print('='*60)
print(f'  Model        : {MODEL_ID}')
print(f'  Generation   : {N_GEN_STEPS} steps  CFG={GEN_CFG}')
print(f'  DDIM steps   : {N_DDIM_STEPS}')
print(f'  Interp steps : {N_INTERP}')
print(f'  z_T_man norm : {z_T_man.norm():.3f}')
print(f'  z_T_wom norm : {z_T_woman.norm():.3f}')
print(f'  PCA var      : {var:.1%}')
print(f'  Output dir   : {os.path.abspath(OUTPUT_DIR)}')
print()
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.isfile(fpath):
        print(f'  {fname:<38} {os.path.getsize(fpath)//1024:>6} KB')
    else:
        print(f'  {fname:<38} [{len(os.listdir(fpath))} files]')
print('='*60)